<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 2B: *Spatial Join Reservoirs*
##### Version Number: 4.0
---
### Contents  
> 1. *Merge sample grid with weather data*
> 2. *Spatial Join of Nearest Sampling Locations with Fire Damage Datasets*
> 3. *Export File*
---
### Notes
- Integrate wildfire impact data with daily weather data from gridMET and date from sampling grid. Sampling grid includes topological, social, inrastructure, and land cover data.
### Inputs
- Daily Weather Readings - `gridmet_final.csv` 
- Wildfire Damage Data - `fire_data.csv` (cleaned in module 1)
- California Mesh Sampling Grid - `Sampling_grids.csv` (built in appendix A) 
---
### Outputs  
- `samples_projected.csv` Cleaned weather dataset merged with fire damage severity and grid data.
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, Polygon

---

### Data Loading and Exploration

In [3]:
samples = pd.read_csv("../data/processed/samples_weather_NDVI.csv")
reservoirs = pd.read_csv("../data/raw/reservoirs.csv")

In [4]:
keep = ['centroid_easting','centroid_northing','Date','grid_id']
join_samples = samples[keep]

## 1. Merge sample points with weather data

#### Add geometry to sampling grid

In [5]:
join_samples['geometry'] = [Point(xy) for xy in zip(join_samples ['centroid_easting'], join_samples ['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(join_samples, geometry='geometry', crs="EPSG:3310")

C:\Users\dusti\AppData\Local\Temp\ipykernel_10224\1994045325.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  join_samples['geometry'] = [Point(xy) for xy in zip(join_samples ['centroid_easting'], join_samples ['centroid_northing'])]


In [6]:
reservoirs['geometry'] = [Point(xy) for xy in zip(reservoirs['Longitude'], reservoirs['Latitude'])]
reservoir_gdf = gpd.GeoDataFrame(reservoirs, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
reservoir_gdf_projected = reservoir_gdf.to_crs(3310)

In [7]:
# Square Buffer radius
buffer_size = 23000

---

## Reservoirs

In [8]:
for dt in reservoir_gdf_projected['Date'].unique():
    
    # Fires on this date
    res_today = reservoir_gdf_projected[reservoir_gdf_projected['Date'] == dt]
    if res_today.empty:
        continue

    # Samples on this date
    samples_today = samples_gdf[samples_gdf['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each sample
    res_buffered = res_today.copy()
    res_buffered['geometry'] = res_buffered['geometry'].apply(lambda p: square_buffer(p, buffer_size))

    # Spatial join: find samples within fire buffers
    joined = gpd.sjoin(samples_today, res_buffered, how='left', predicate='intersects')
    
    joined['level_missing'] = 1 - joined['Reservoir Level_has_data']
    
    # Aggregate counts and total damage per sample
    grouped = joined.groupby('grid_id').agg(
        reservoir_count=('Station_ID', 'count'),
        total_reservoir_level=('Reservoir Level', 'sum'),
        stations_missing_levels = ('level_missing', 'sum')
    ).fillna(0)

    # Assign values back to main dataframe
    samples_gdf.loc[samples_gdf['Date'] == dt, 'reservoir_count'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['reservoir_count']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'total_reservoir_level'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['total_reservoir_level']).fillna(0)
    samples_gdf.loc[samples_gdf['Date'] == dt, 'stations_missing_levels'] = \
        samples_gdf.loc[samples_gdf['Date'] == dt, 'grid_id'].map(grouped['stations_missing_levels']).fillna(0)

In [9]:
post_merge_check(samples_gdf, join_samples)

Premerged shape:  (608880, 5)
Merged shape:  (608880, 8)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


In [10]:
samples_gdf.isna().sum()

centroid_easting           0
centroid_northing          0
Date                       0
grid_id                    0
geometry                   0
reservoir_count            0
total_reservoir_level      0
stations_missing_levels    0
dtype: int64

In [11]:
samples_gdf = samples_gdf.drop(columns=['centroid_easting','centroid_northing','geometry'])

---

In [12]:
# Merge on BOTH station and date
samples_res = samples.merge(
    samples_gdf,
    on=['grid_id','Date'],
    how='left'
)

In [13]:
post_merge_check(samples_res, samples)

Premerged shape:  (608880, 65)
Merged shape:  (608880, 68)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0


NA values after merge:  0


In [14]:
samples_res.isna().sum()

Sample_ID                         0
Date                              0
Burning Index                     0
Energy Release Component          0
Actual Evapotranspiration         0
                                 ..
NDVI_mean_difference              0
NDVI_mean_difference_has_value    0
reservoir_count                   0
total_reservoir_level             0
stations_missing_levels           0
Length: 68, dtype: int64

## 3. Export File

In [15]:
samples_res.to_csv('../data/processed/samples_res.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
